# Buổi 8 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `tuong_quan.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Chạy code đầu buổi (chạy lại ô này sau bước 2, 4, 5)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tuong_quan as tq

kt = tq.doc_kinh_te()
df = tq.ghep_tai_nhiet()
print("1. CPI × dân số:", tq.tuong_quan_chuoi(kt["cpi"], kt["dan_so"]))
print("3. độ trễ dẫn dắt (thô):", tq.do_tre_dan_dat(df["cdd"], df["tai"], bac=None)["tre"],
      "| sau prewhitening:", tq.do_tre_dan_dat(df["cdd"], df["tai"])["tre"])
print("5.", tq.granger_hai_chieu(pd.Series(df["cdd"].to_numpy(), index=df.index), df["tai"])["ket_luan"])

## Bước 2 — Tương quan giả

Sửa `tuong_quan_chuoi` (tài liệu mục 4.2), rồi chạy lại ô bước 1. Chấm: `python lab.py check` trong terminal.

## Bước 3 — Phi tuyến và MI (mục 4.3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sc = ax.scatter(df["nhiet"], df["tai"] / 1000, c=df.index.month, s=2, cmap="twilight")
ax.set_xlabel("nhiệt độ (°C)")
ax.set_ylabel("tải (GW)")
plt.colorbar(sc, label="tháng")

y = df["tai"].to_numpy()
X = np.column_stack([np.ones(len(df)), df["cdd"], df["hdd"]])
du = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
print("R² theo nhiệt độ:", round(tq.hoi_quy_don(df["nhiet"], y)["r2"], 3), "| theo CDD + HDD:", round(1 - du.var() / y.var(), 3))
print(tq.tuong_quan(df["nhiet"], y))
print("MI và hoán vị theo khối 168 giờ:", tq.kiem_y_nghia_mi(df["nhiet"], y, do_dai_khoi=168, so_lan=100))

## Bước 4 — Tương quan chéo và prewhitening (mục 4.4)

Sửa `do_tre_dan_dat` để lọc bằng `loc_prewhiten` khi `bac` khác `None`, rồi chạy lại ô này và ô bước 1.

In [ ]:
x = np.array([1, 2, 1, 2, 3, 4, 3, 2, 1, 2, 1, 0.0])
y = np.r_[np.ones(2), x[:-2]]
print("ví dụ tay, thô      :", np.round(tq.ccf_tu_viet(x, y, 4), 2))
print("ví dụ tay, sai phân :", np.round(tq.ccf_tu_viet(np.diff(x), np.diff(y), 4), 2))

rng = np.random.default_rng(1)
gia = rng.normal(size=1000)
print("chuỗi giả y = x trễ 3, đỉnh ở:", int(np.argmax(tq.ccf_tu_viet(gia, np.r_[np.zeros(3), gia[:-3]], 8))))

tho = tq.do_tre_dan_dat(df["cdd"].to_numpy(), df["tai"].to_numpy(), so_tre=30, bac=None)
sach = tq.do_tre_dan_dat(df["cdd"].to_numpy(), df["tai"].to_numpy(), so_tre=30)
print("trễ 0, 1, 24 — thô:", np.round(tho["ccf"][[0, 1, 24]], 3), "| sau lọc:", np.round(sach["ccf"][[0, 1, 24]], 3))

## Bước 5 — Tương quan trượt và Granger (mục 4.5, 4.6)

Sửa `granger_hai_chieu`, rồi chạy lại ô bước 1.

In [ ]:
ngay = df.resample("D").mean()
ax = pd.DataFrame({f"cửa sổ {w} ngày": tq.tuong_quan_truot(ngay["nhiet"], ngay["tai"], w) for w in (30, 90)}).plot(figsize=(8, 3.5))
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("tương quan nhiệt độ × tải")
print(tq.granger_hai_chieu(pd.Series(df["cdd"].to_numpy(), index=df.index), df["tai"]))